In [3]:
# ============================================================
# Notebook 4: 04_sensitivity_analysis.ipynb  (UNIFIED)
# Purpose : Robustness of the double-exclusion finding to the
#           group-definition cutoffs. Same all-continuous DiCE
#           config as Notebook 2.
# ============================================================

import os, time, pickle, logging
import numpy as np
import pandas as pd
from scipy.stats import beta
import dice_ml
from dice_ml import Dice

# ------------------------------------------------------------
# 0. Load Notebook 1 artifacts
# ------------------------------------------------------------
DATA_DIR = r"C:\Users\miy\Downloads\1000_논문투고\117.금융 합성데이터\3.개방데이터\1.데이터\1. 합성데이터"
ARTIFACT_DIR = os.path.join(DATA_DIR, "_artifacts")
with open(os.path.join(ARTIFACT_DIR, "nb1_artifacts.pkl"), "rb") as f:
    A = pickle.load(f)

rf         = A["rf"]
X_train    = A["X_train"]; y_train = A["y_train"]
X_test     = A["X_test"]
FEATURES   = A["FEATURES"]; IMMUTABLE = A["IMMUTABLE"]
ACTIONABLE = A["ACTIONABLE"]; TARGET  = A["TARGET"]

logging.getLogger("dice_ml").setLevel(logging.ERROR)
try:
    import tqdm; from functools import partialmethod
    tqdm.tqdm.__init__ = partialmethod(tqdm.tqdm.__init__, disable=True)
except Exception:
    pass

# ------------------------------------------------------------
# 1. Encoded axis constants (from Notebook 1 encoding)
#    age ordinal: 20대=0 ... 70대이상=5 ; gender coded {1,2}->{0,1}
# ------------------------------------------------------------
AGE_CODE = {"20대":0,"30대":1,"40대":2,"50대":3,"60대":4,"70대이상":5}
FEMALE_CODE, MALE_CODE = 1, 0
YOUNG = [AGE_CODE["30대"]]

test = X_test.copy()
test["proba"] = rf.predict_proba(X_test[FEATURES])[:, 1]
test["pred"]  = rf.predict(X_test[FEATURES])
tq = X_test["mn_credit_tenure"]

def make_groups(df_enc, old_age_codes, young_age_codes, tenure_short, tenure_long):
    g = pd.Series("other", index=df_enc.index)
    adv = (df_enc["age"].isin(young_age_codes) &
           (df_enc["code_gender"] == MALE_CODE) &
           (df_enc["mn_credit_tenure"] >= tenure_long))
    dis = (df_enc["age"].isin(old_age_codes) &
           (df_enc["code_gender"] == FEMALE_CODE) &
           (df_enc["mn_credit_tenure"] <= tenure_short))
    g[adv] = "advantaged"; g[dis] = "disadvantaged"
    return g

# ------------------------------------------------------------
# 2. DiCE setup — SAME all-continuous config as Notebook 2
# ------------------------------------------------------------
ALL_CONTINUOUS = FEATURES[:]
train_for_dice = X_train.copy(); train_for_dice[TARGET] = y_train.values
d = dice_ml.Data(dataframe=train_for_dice, continuous_features=ALL_CONTINUOUS, outcome_name=TARGET)
m = dice_ml.Model(model=rf, backend="sklearn")
exp = Dice(d, m, method="random")

ACTION_HORIZON = 0.20
def per_feature_ranges(row):
    rng = {}
    for f in ACTIONABLE:
        v = float(row[f]); lo, hi = sorted([v*(1-ACTION_HORIZON), v*(1+ACTION_HORIZON)])
        rng[f] = [lo, hi]
    return rng
def check_path(orig_row, cf_row):
    imm = any(not np.isclose(float(orig_row[f]), float(cf_row[f])) for f in IMMUTABLE)
    act = False
    for f in ACTIONABLE:
        o=float(orig_row[f]); c=float(cf_row[f])
        act |= (c!=0) if o==0 else abs(c-o)/abs(o) > ACTION_HORIZON+1e-9
    return imm, act

# ------------------------------------------------------------
# 3. Cutoff sweep sets
# ------------------------------------------------------------
TENURE_SETS = {
    "p25_p75": (tq.quantile(0.25), tq.quantile(0.75)),
    "p20_p80": (tq.quantile(0.20), tq.quantile(0.80)),
    "p33_p67": (tq.quantile(0.33), tq.quantile(0.67)),
}
AGE_SETS = {
    "old_60+": [AGE_CODE["60대"], AGE_CODE["70대이상"]],
    "old_50+": [AGE_CODE["50대"], AGE_CODE["60대"], AGE_CODE["70대이상"]],
    "old_70+": [AGE_CODE["70대이상"]],
}

# ------------------------------------------------------------
# 4. Axis 1 — CDR sensitivity (fast, all 9 combos)
# ------------------------------------------------------------
cdr_rows = []
for tname, (ts, tl) in TENURE_SETS.items():
    for aname, old_codes in AGE_SETS.items():
        g = make_groups(test, old_codes, YOUNG, ts, tl)
        pa = test.loc[g=="advantaged","proba"].mean()
        pdd= test.loc[g=="disadvantaged","proba"].mean()
        cdr_rows.append({"tenure":tname,"age":aname,
                         "n_adv":int((g=="advantaged").sum()),
                         "n_dis":int((g=="disadvantaged").sum()),
                         "p_adv_%":round(100*pa,2),"p_dis_%":round(100*pdd,2),
                         "CDR":round(pa/pdd,1) if pdd>0 else np.inf})
print("=== Axis 1: CDR sensitivity ===")
print(pd.DataFrame(cdr_rows).to_string(index=False))

# ------------------------------------------------------------
# 5. Axis 2 — recourse reliability sensitivity (3 representative combos)
# ------------------------------------------------------------
def reliable_rr(subset):
    n = len(subset); rel = 0
    for _, row in subset.iterrows():
        q = row[FEATURES].to_frame().T.astype(float)
        try:
            cf = exp.generate_counterfactuals(q, total_CFs=4, desired_class=1,
                    features_to_vary=ACTIONABLE, permitted_range=per_feature_ranges(row),
                    verbose=False)
            cfdf = cf.cf_examples_list[0].final_cfs_df
            if cfdf is not None and len(cfdf):
                for _, c in cfdf[FEATURES].iterrows():
                    iv, av = check_path(row, c)
                    if not iv and not av: rel += 1; break
        except Exception:
            pass
    return rel, n

SENS_N = 120
REP_COMBOS = [
    ("p20_p80","old_70+", tq.quantile(0.20), tq.quantile(0.80), [AGE_CODE["70대이상"]]),
    ("p25_p75","old_60+", tq.quantile(0.25), tq.quantile(0.75), [AGE_CODE["60대"], AGE_CODE["70대이상"]]),
    ("p33_p67","old_50+", tq.quantile(0.33), tq.quantile(0.67), [AGE_CODE["50대"], AGE_CODE["60대"], AGE_CODE["70대이상"]]),
]

sens_rows = []
for tname, aname, ts, tl, old_codes in REP_COMBOS:
    g = make_groups(test, old_codes, YOUNG, ts, tl)
    for grp in ["advantaged", "disadvantaged"]:
        rej = test[(g == grp) & (test["pred"] == 0)]
        s = rej.sample(n=min(SENS_N, len(rej)), random_state=42)
        t0 = time.time()
        k, n = reliable_rr(s)
        sens_rows.append({"tenure":tname,"age":aname,"group":grp,
                          "n":n,"n_reliable":k,
                          "Reliable_RR_%":round(100*k/n,2) if n else np.nan})
        print(f"  [{tname}/{aname}/{grp}] n={n} reliable={k} ({time.time()-t0:.0f}s)")

print("\n=== Axis 2: recourse reliability sensitivity ===")
print(pd.DataFrame(sens_rows).to_string(index=False))

=== Axis 1: CDR sensitivity ===
 tenure     age  n_adv  n_dis  p_adv_%  p_dis_%  CDR
p25_p75 old_60+    739    822    36.77     2.14 17.2
p25_p75 old_50+    739   2304    36.77     2.59 14.2
p25_p75 old_70+    739    211    36.77     1.59 23.1
p20_p80 old_60+    401    660    39.31     1.92 20.5
p20_p80 old_50+    401   1863    39.31     2.16 18.2
p20_p80 old_70+    401    165    39.31     1.51 26.0
p33_p67 old_60+   1347   1042    35.82     2.66 13.5
p33_p67 old_50+   1347   2905    35.82     3.40 10.5
p33_p67 old_70+   1347    270    35.82     2.26 15.9
No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec
No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec
No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec
No Counterfactuals found for the given configuratio